<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day One: From Brain Images to a First fMRI Comparison

Today we will use Python to open brain images, inspect the numbers inside them, and ask a simple question of real fMRI data.

By the end, you will be able to:

- describe voxels, slices, volumes, and 4D fMRI data,
- inspect an MRI image as an array of numbers,
- navigate through the brain with different plots,
- explain why an fMRI image needs to be compared with something,
- describe the basic idea of a GLM and a contrast, and
- calculate a first-level contrast for one participant.

## What is a Jupyter notebook?

A Jupyter notebook is a page where text, code, pictures, and results can live together.

This notebook has two kinds of boxes, called **cells**:

- **Text cells**, like this one, explain what is happening.
- **Code cells** contain Python instructions that the computer can run.

To run a code cell, click the play button on the left. You can also click inside the cell and press **Shift + Enter**.

Run the cells from top to bottom. A later cell may need a variable that was created in an earlier cell.

## Setup: press play here first

The next cell installs and imports our tools. It also loads a standard anatomical brain template.

You do not need to understand every setup line yet. Run it once and wait for **Setup complete**.

In [ ]:
# @title Setup: install tools and load an anatomical template
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import FirstLevelModel, spm_hrf

%matplotlib inline

# This is an average anatomical brain in a shared reference space.
anat_img = datasets.load_mni152_template(resolution=2)

print("Setup complete.")
print("Anatomical image shape:", anat_img.shape)

## MRI data: anatomy and function

MRI scanners can create different kinds of data. We will use two today:

- An **anatomical MRI** is a detailed 3D picture of brain structure. Each voxel has one brightness value.
- **fMRI** records many 3D brain volumes, one after another. Each voxel therefore has a series of values over time.

A scanner collects the brain in **slices**. A stack of slices makes one 3D **volume**. In fMRI, many volumes make a 4D data set.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/mri_fmri_build_up.png" alt="MRI and fMRI data built from voxels, slices, volumes, and time" width="850">

The highlighted green cube is one **voxel**. A voxel is like a 3D pixel.

## A brain image is an array of numbers

A computer does not begin with a picture of a brain. It stores a grid of numbers called an **array**. Plotting software turns those numbers into brightness or color.

For a 3D anatomical image, the array has three dimensions:

- the first position moves through the **x** direction,
- the second moves through the **y** direction, and
- the third moves through the **z** direction.

The image's `shape` tells us how many voxel positions exist along each dimension.

In [ ]:
# Turn the image data into a NumPy array that we can inspect.
anat_data = anat_img.get_fdata()

print("Python object:", type(anat_img))
print("Array type:", type(anat_data))
print("Array shape:", anat_data.shape)
print("Number of dimensions:", anat_data.ndim)

### Reading one voxel

We can use three array positions to ask for one voxel value. Python starts counting at **0**, not 1.

The code below finds the middle array position automatically. The three numbers are an **array index**. They identify a stored voxel, but they are not yet millimetre coordinates on a brain map.

In [ ]:
# Find the middle index along x, y, and z.
middle_x = anat_data.shape[0] // 2
middle_y = anat_data.shape[1] // 2
middle_z = anat_data.shape[2] // 2

middle_value = anat_data[middle_x, middle_y, middle_z]

print("Middle array index:", (middle_x, middle_y, middle_z))
print("Value stored there:", middle_value)

An array can also give us a small block of nearby voxels. The colon in `a:b` means: start at `a` and stop just before `b`.

In [ ]:
# Select a 3 by 3 by 3 block around the middle.
small_block = anat_data[
    middle_x - 1:middle_x + 2,
    middle_y - 1:middle_y + 2,
    middle_z - 1:middle_z + 2,
]

print("Block shape:", small_block.shape)
print(small_block)

### Turn the numbers into a 3D object

The array above contains 27 numbers arranged as `3 × 3 × 3`. The next cell draws those same 27 positions as cubes.

Each cube is one voxel. Its color comes from the number stored at that position: darker purple means a lower value and brighter yellow means a higher value.

In [ ]:
# Scale the 27 values between 0 and 1 so they can become colors.
block_range = np.ptp(small_block)
scaled_values = (small_block - small_block.min()) / block_range
voxel_colors = plt.cm.viridis(scaled_values)

# Draw all 27 array positions as filled cubes.
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
ax.voxels(
    np.ones(small_block.shape, dtype=bool),
    facecolors=voxel_colors,
    edgecolor="black",
)
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.set_zlabel("z position")
ax.set_title("The 3 × 3 × 3 array drawn as 27 voxels")
ax.set_box_aspect((1, 1, 1))
plt.show()

## Brain slices and directions

Scientists view the 3D array as slices from three directions:

- **Sagittal** slices divide left and right. Nilearn calls this the `x` direction.
- **Coronal** slices divide front and back. Nilearn calls this the `y` direction.
- **Axial** slices divide top and bottom. Nilearn calls this the `z` direction.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/AnatomicalPlanes.png" alt="Sagittal, coronal, and axial brain planes" width="700">

## Plot the anatomical image

The default `ortho` view shows one sagittal, one coronal, and one axial slice together. The crosshairs mark the same location in all three views.

In [ ]:
plotting.plot_anat(
    anat_img,
    display_mode="ortho",
    title="Anatomical MRI: three slice directions",
)
plotting.show()

Sometimes we want to compare several slices from the same direction. Here, `display_mode="z"` asks for axial slices, and `cut_coords` chooses their heights in millimetres.

These map coordinates are different from the array indices we used earlier. Nilearn reads the image information needed to place the voxels in brain space.

In [ ]:
slice_heights = [-30, -10, 10, 30, 50, 70]

plotting.plot_anat(
    anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    title="Axial slices from lower to higher positions",
)
plotting.show()

## Explore the brain interactively

`view_img` creates a viewer you can click. Try clicking in each view and watch how the crosshair and the `x`, `y`, and `z` coordinates change.

In [ ]:
anat_viewer = plotting.view_img(
    anat_img,
    bg_img=False,
    cmap="gray",
    colorbar=False,
    symmetric_cmap=False,
    title="Click to explore the anatomical template",
)

anat_viewer

### Brain-region treasure hunt

Later, you will study **vision**, **audition**, or **language**. Use the interactive viewer to navigate near these approximate landmarks. The coordinates are clues, not exact borders: brain areas are larger than one point and differ between people.

1. **Vision:** move toward `(x=0, y=-85, z=10)`. Are you near the front or the back of the brain? This is near visual cortex in the occipital lobe.
2. **Audition:** move toward `(x=-50, y=-20, z=10)`. This is near auditory cortex in the upper temporal lobe, above the left ear.
3. **Language:** move toward `(x=-45, y=20, z=20)`. This is near a left frontal region often involved in producing language.

For each landmark, describe its location using ordinary words such as **left/right**, **front/back**, and **top/bottom**. Do not worry about finding one perfect voxel.

## fMRI adds a fourth dimension: time

An anatomical image has a value at every `(x, y, z)` voxel position. An fMRI image adds a fourth position, `t`, for **time**.

We can think about the same 4D data in two ways:

1. a sequence of 3D brain volumes, or
2. a time series from every voxel.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_4d_timeseries.png" alt="A sequence of fMRI volumes and a time series from one voxel" width="700">

The **repetition time**, often shortened to **TR**, is the time between one whole-brain volume and the next.

### What is the fMRI signal?

The scanner stores a signal-intensity number for each voxel at each time point. That number is influenced by the **BOLD signal**: changes in blood oxygen that are connected to brain activity.

The line from one voxel can rise and fall because of task-related activity, breathing, heartbeat, head movement, scanner noise, and slow changes during the scan.

So a bump in one raw time series is **not automatically brain activation**. We need to compare the measured signal with what happened during the experiment.

## The localizer experiment

We will use a real teaching data set from one participant. During a short scan, the participant completed several small tasks:

| Theme | Example conditions | Possible question |
|---|---|---|
| Vision | horizontal and vertical checkerboards | Which voxels respond while looking at patterns? |
| Audition | listening to sentences or calculations | Which voxels respond while listening? |
| Language | reading and listening to sentences | How do reading and listening differ? |
| Movement | left- and right-hand button presses | How do the two hands differ? |
| Calculation | visual and audio calculations | Does presentation by sight or sound matter? |

The data set contains:

- one preprocessed 4D fMRI image, and
- an **events table** that records what happened and when.

The next cell downloads the data the first time it is run. This may take a minute.

In [ ]:
# Download one participant's fMRI image and events table.
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("fMRI image shape:", fmri_img.shape)
print("Number of dimensions:", len(fmri_img.shape))
print("Number of recorded events:", len(events))

The first three shape numbers count voxel positions in space. The last number counts time points.

A shape of `(53, 63, 46, 128)` means 53 by 63 by 46 spatial positions, measured at 128 time points.

### What happened during the scan?

Each row of the events table describes one event:

- `trial_type`: what the participant did,
- `onset`: when it began, in seconds after the scan started,
- `duration`: how long it lasted, in seconds.

In [ ]:
# Display the first ten events in the experiment.
events.head(10)

In [ ]:
# List every condition that the model can compare.
for condition in sorted(events["trial_type"].unique()):
    print(condition)

## Look at the fMRI image

A plotting function needs a 3D image, so we can select one time point with `index_img`. We can also use `mean_img` to average all time points into one 3D image.

The fMRI image looks blurrier and less detailed than the anatomical image.

In [ ]:
first_volume = image.index_img(fmri_img, 0)

plotting.plot_epi(
    first_volume,
    title="The first 3D volume in the fMRI scan",
)
plotting.show()

In [ ]:
mean_fmri = image.mean_img(fmri_img, copy_header=True)

plotting.plot_epi(
    mean_fmri,
    title="Mean across all fMRI time points",
)
plotting.show()

### Your turn: make the mean image interactive

The code needs only the mean image we created and Nilearn's `view_img` function. Run it, then click through the brain.

In [ ]:
# Create an interactive view of the mean fMRI image.
mean_fmri_viewer = plotting.view_img(
    mean_fmri,
    bg_img=False,
    cmap="gray",
    symmetric_cmap=False,
    title="Mean fMRI signal",
)

mean_fmri_viewer

### What do these colors mean?

In this mean image, brighter and darker colors show the **average signal intensity** recorded at each voxel.

They do **not** tell us which areas were activated by a task. A voxel can have a high average signal for reasons related to tissue, scanner sensitivity, and image position.

To talk about task-related activity, we need a comparison: **more signal during what, compared with what?**

## Extract one voxel's time series

To select one voxel's full time series, we choose one `x`, `y`, and `z` position and keep every time point with `:`:

```python
fmri_data[x, y, z, :]
```

In [ ]:
# Choose the middle array position in the fMRI image.
voxel_x = fmri_img.shape[0] // 2
voxel_y = fmri_img.shape[1] // 2
voxel_z = fmri_img.shape[2] // 2

# Read this voxel at every time point.
voxel_signal = np.asarray(
    fmri_img.dataobj[voxel_x, voxel_y, voxel_z, :]
)

print("Voxel index:", (voxel_x, voxel_y, voxel_z))
print("Time-series shape:", voxel_signal.shape)
print("First 10 signal values:", voxel_signal[:10])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue")
plt.xlabel("Time point (volume number)")
plt.ylabel("fMRI signal intensity")
plt.title("Signal over time from one voxel")
plt.grid(alpha=0.25)
plt.show()

## Before analysis: preparing the data

Raw fMRI data cannot usually be compared immediately. **Preprocessing** prepares the images so that the time points and participants line up more sensibly. Common steps correct for head movement, align images, place brains in a shared space, and reduce some image noise.

The localizer image supplied by Nilearn has already been preprocessed. Good analysis also includes **quality checks**: for example, checking whether movement was too large or alignment worked correctly.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_analysis_steps.png" alt="Preprocessing followed by first-level and second-level analysis, with quality checks" width="850">

- **First-level analysis** asks a question within one participant's data.
- **Second-level analysis** combines first-level results from several participants to ask what is consistent across the group.

Today we will do one small **first-level analysis**.

## A simple model of the experiment

A **general linear model**, or **GLM**, asks how well different expected patterns explain the measured signal at each voxel.

- A **predictor** or **regressor** is an expected signal pattern. We make one for each experimental condition.
- The **design matrix** stores these predictors. Rows are time points and columns are predictors.
- The model estimates how strongly each predictor is connected to each voxel's measured time series.

The BOLD response is delayed. It usually rises several seconds after an event and then slowly returns toward its starting level. The model uses a curve called the **hemodynamic response function**, or **HRF**, to include this delay.

In [ ]:
# Draw the HRF shape used to model a short event.
hrf_step = 0.1
hrf = spm_hrf(hrf_step, oversampling=1, time_length=30)
hrf_time = np.arange(len(hrf)) * hrf_step

plt.figure(figsize=(8, 3))
plt.plot(hrf_time, hrf, color="darkorange", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Seconds after an event")
plt.ylabel("Expected BOLD response")
plt.title("The HRF: the BOLD signal responds slowly")
plt.grid(alpha=0.25)
plt.show()

### Fit the first-level model

We give the model three important pieces of information:

1. the 4D fMRI image,
2. the events table, and
3. the TR, which is 2.4 seconds for this scan.

Nilearn then builds the predictors and fits the model separately at every voxel.

In [ ]:
# Describe and fit one participant's first-level model.
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=True,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Model fitted successfully.")

### Inspect the design matrix

The colored columns are the model's predictors. The condition columns show when the model expects a BOLD response. The additional drift and constant columns help account for slow background patterns that are not our main question.

In [ ]:
design_matrix = first_level_model.design_matrices_[0]

plotting.plot_design_matrix(design_matrix)
plt.title("Design matrix: expected patterns over time")
plt.show()

print("Design matrix shape:", design_matrix.shape)

## A contrast: right hand compared with left hand

A **contrast** is a comparison between model estimates.

Our question is:

> Where was the modeled response stronger for right-hand button presses than for left-hand button presses?

In short:

```text
right-hand response - left-hand response
```

Subtracting matters. A colorful map of one condition alone cannot tell us whether the response is special to that condition.

In [ ]:
# Compare visual right-hand presses with visual left-hand presses.
right_minus_left = (
    "visual_right_hand_button_press "
    "- visual_left_hand_button_press"
)

right_minus_left_map = first_level_model.compute_contrast(
    right_minus_left,
    output_type="z_score",
)

print("Contrast calculated.")

### Plot the contrast map

The map below shows only voxels with an absolute z score above 3 so that the pattern is easier to see.

- Warm colors show a stronger modeled response for the **right hand** than the left hand.
- Cool colors show a stronger modeled response for the **left hand** than the right hand.

Because each side of the brain mainly controls the opposite side of the body, we expect much of the hand difference to appear on opposite sides of the motor system.

This threshold is for a classroom demonstration. It is **not** a full test of statistical significance and does not correct for the many thousands of voxels examined.

In [ ]:
plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=mean_fmri,
    display_mode="z",
    cut_coords=7,
    threshold=3.0,
    symmetric_cbar=True,
    title="Right-hand button press minus left-hand button press",
)
plotting.show()

### Read the result carefully

Discuss these questions:

1. Which side contains most of the warm-colored voxels?
2. Which side contains most of the cool-colored voxels?
3. Why does the title need the word **minus**?
4. What could we compare for a project about vision, audition, or language?

Possible next contrasts include checkerboards versus another condition, sentence listening versus a visual condition, or sentence reading versus a non-language visual condition. Choosing a fair comparison is part of designing the scientific question.

## Sources and further reading

This notebook adapts teaching material and figures from `TEWA2/02_Understanding_MRI_Data.ipynb` and `TEWA2/06_First_Level_Analysis.ipynb` in this repository.

- [Nilearn: 3D and 4D images](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_3d_and_4d_niimg.html)
- [Nilearn: first-level models](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)
- Pinel et al. (2007), the example localizer fMRI data set: https://doi.org/10.1186/1471-2202-8-91